### Tool-calling Agent

This project is a **Databricks-native application** and requires a Databricks workspace to run. 

This is a modification of a Databricks auto-generated notebook created using the Databricks UI. This notebook does the following:
- Authors a tool-calling [MLflow's `ResponsesAgent`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ResponsesAgent) that uses the OpenAI client
- Manually tests the agent's output
- Evaluates the agent with Mosaic AI Agent Evaluation
- Logs and deploys the agent

In [0]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents mlflow-skinny[databricks]
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


#### Define the agent in code
Agent code is defined in a single cell, enabling easily writing it to a local Python file for subsequent logging and deployment using the `%%writefile` command.

In [0]:
%%writefile agent.py
import json
from typing import Any, Callable, Generator, Optional
from uuid import uuid4
import warnings

import backoff
import mlflow
import openai
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI, UCFunctionToolkit, VectorSearchRetrieverTool
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from openai import OpenAI
from pydantic import BaseModel
from unitycatalog.ai.core.base import get_uc_function_client

############################################
# Define LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "system.ai.meta-llama-3-3-70b-instruct"

SYSTEM_PROMPT = """You are a marketing analytics assistant for an online supermarket. You answer questions about customer behaviour using the available tools.

GENERAL RULES:
- Always call a tool before answering any question about the data. Never state numbers from memory or invent them.
- The currency is NOT USD.  Do NOT use currency symbols at all
- If no tool can answer the question, say so plainly. Do not guess.
- demographic_profile, channel_mix_by_demographic and deal_affinity_by_demographic take one of exactly these values: Education, Marital_Status, Age_category,
  Income_category.
- The Education levels are Brazilian, and the categories correspond roughly to the following years of education: Basic: 12, Graduation: 16, 2n Cycle:18, Master: 18, PhD: 22 
- Ratios are proportions between 0 and 1 (e.g. 0.35 = 35% of purchases).
- When you report figures, name the source tool and the group it refers to.
- Keep answers under 150 words unless asked for detail. Use plain English, not statistical jargon.
  
  RFM SEGMENT DEFINITIONS:
  - rfm_customers_in_segment takes one of exactly these eight segment names: Champions, High-potential growth, Loyal, Recents, Critical retention, Low-priority churn, High value at risk, Standard base.
- rfm_customers_by_score_band takes score_component of R, F or M. Valid scores: R = recency score 1-3, F/M = frequency/monetary 1-5.  Higher = better.
- R=3 (recently active), R=2 (cooling), R=1 (lapsed).
- Champions: recent + frequent + high spend. Protect, upsell.
- High-potential growth: recent, strong on one of F/M but not both. Nurture toward Champions.
- Loyal: recent and frequent, lower spend. Increase basket value.
- Recents: recent but low frequency and monetary. Onboard, build habit.
- High value at risk: WAS frequent and high-value, now cooling (R=2). Priority win-back -  relationship still recoverable.
- Critical retention: identical value profile to High value at risk but already lapsed (R=1).  Last-chance offer only; assume recovery cost is higher.
- Standard base: average behaviour across all dimensions. Baseline comms only.
- Low-priority churn: lapsed AND low value. Do not allocate campaign budget.

RFM-SPECIFIC RULES:
- Segments sharing the same F/M bands differ ONLY on recency. Never treat them as interchangeable - the recency gap is the entire commercial distinction.
- When two segments have similar monetary value, prioritise by recency: retaining an actively-cooling customer costs less than reactivating a lapsed one.
 - When comparing segments of similar value, explicitly state your recency-based prioritisation. 
- Never recommend acquisition spend while High value at risk remains untargeted."""

###############################################################################
## Define tools for agent
###############################################################################
class ToolInfo(BaseModel):
    """
    Class representing a tool for the agent.
    - "name" (str): The name of the tool.
    - "spec" (dict): JSON description of the tool (matches OpenAI Responses format)
    - "exec_fn" (Callable): Function that implements the tool logic
    """

    name: str
    spec: dict
    exec_fn: Callable


def create_tool_info(tool_spec, exec_fn_param: Optional[Callable] = None):
    tool_spec["function"].pop("strict", None)
    tool_name = tool_spec["function"]["name"]
    udf_name = tool_name.replace("__", ".")

    # Define a wrapper that accepts kwargs for the UC tool call,
    # then passes them to the UC tool execution client
    def exec_fn(**kwargs):
        function_result = uc_function_client.execute_function(udf_name, kwargs)
        if function_result.error is not None:
            return function_result.error
        else:
            return function_result.value
    return ToolInfo(name=tool_name, spec=tool_spec, exec_fn=exec_fn_param or exec_fn)


TOOL_INFOS = []

UC_TOOL_NAMES = ["analytics.marketing.channel_mix_by_rfm_segment", "analytics.marketing.rfm_customers_by_score_band", "analytics.marketing.rfm_customers_in_segment", "analytics.marketing.rfm_segment_summary", "analytics.marketing.demographic_profile", "analytics.marketing.deal_affinity_by_demographic", "analytics.marketing.channel_mix_by_demographic"]

uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
uc_function_client = get_uc_function_client()
for tool_spec in uc_toolkit.tools:
    TOOL_INFOS.append(create_tool_info(tool_spec))

class ToolCallingAgent(ResponsesAgent):
    """
    Class representing a tool-calling Agent
    """

    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        """Initializes the ToolCallingAgent with tools."""
        self.llm_endpoint = llm_endpoint
        self.model_serving_client: OpenAI = DatabricksOpenAI(use_ai_gateway=True)
        self._tools_dict = {tool.name: tool for tool in tools}

    def get_tool_specs(self) -> list[dict]:
        """Returns tool specifications in the format OpenAI expects."""
        return [tool_info.spec for tool_info in self._tools_dict.values()]

    @mlflow.trace(span_type=SpanType.TOOL)
    def execute_tool(self, tool_name: str, args: dict) -> Any:
        """Executes the specified tool with the given arguments."""
        return self._tools_dict[tool_name].exec_fn(**args)

    @staticmethod
    def _merge_consecutive_assistant_messages(cc_messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
        # The canonical OpenAI shape for parallel tool calls is one assistant
        # message whose tool_calls array lists every call in that turn.
        # to_chat_completions_input emits a separate assistant message per
        # function_call, which some providers reject. Collapse consecutive
        # assistant messages so all tool_calls from one turn share a single
        # message.
        merged: list[dict[str, Any]] = []
        for msg in cc_messages:
            if (
                msg.get("role") == "assistant"
                and merged
                and merged[-1].get("role") == "assistant"
            ):
                prev = merged[-1]
                prev_text = prev.get("content") if prev.get("content") not in (None, "", "tool call") else None
                cur_text = msg.get("content") if msg.get("content") not in (None, "", "tool call") else None
                if prev_text and cur_text:
                    prev["content"] = prev_text + "\n" + cur_text
                elif cur_text:
                    prev["content"] = cur_text
                if msg.get("tool_calls"):
                    prev["tool_calls"] = (prev.get("tool_calls") or []) + msg["tool_calls"]
            else:
                merged.append(dict(msg))
        return merged

    def call_llm(self, messages: list[dict[str, Any]]) -> Generator[dict[str, Any], None, None]:
        cc_messages = self._merge_consecutive_assistant_messages(to_chat_completions_input(messages))
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="PydanticSerializationUnexpectedValue")
            for chunk in self.model_serving_client.chat.completions.create(
                model=self.llm_endpoint,
                messages=cc_messages,
                tools=self.get_tool_specs(),
                stream=True,
            ):
                chunk_dict = chunk.to_dict()
                if len(chunk_dict.get("choices", [])) > 0:
                    yield chunk_dict

    def handle_tool_call(
        self,
        tool_call: dict[str, Any],
        messages: list[dict[str, Any]],
    ) -> ResponsesAgentStreamEvent:
        """
        Execute tool calls, add them to the running message history, and return a ResponsesStreamEvent w/ tool output
        """
        try:
            args = json.loads(tool_call.get("arguments"))
        except Exception as e:
            args = {}
        result = str(self.execute_tool(tool_name=tool_call["name"], args=args))

        tool_call_output = self.create_function_call_output_item(tool_call["call_id"], result)
        messages.append(tool_call_output)
        return ResponsesAgentStreamEvent(type="response.output_item.done", item=tool_call_output)

    def call_and_run_tools(
        self,
        messages: list[dict[str, Any]],
        max_iter: int = 10,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        for _ in range(max_iter):
            # The LLM may emit multiple tool calls in a single turn (parallel
            # tool calls). The next LLM request must include a tool_result for
            # every tool_use emitted in the previous turn; missing any one
            # causes the request to be rejected. Before going back to the LLM,
            # execute every function_call whose call_id has no matching
            # function_call_output yet.
            handled = {m["call_id"] for m in messages if m.get("type") == "function_call_output"}
            pending = [m for m in messages if m.get("type") == "function_call" and m["call_id"] not in handled]
            if pending:
                for call in pending:
                    yield self.handle_tool_call(call, messages)
                continue

            last_msg = messages[-1]
            if last_msg.get("type") == "message" and last_msg.get("role") == "assistant":
                return

            yield from output_to_responses_items_stream(
                chunks=self.call_llm(messages), aggregator=messages
            )

        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item("Max iterations reached. Stopping.", str(uuid4())),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(
                metadata={
                    "mlflow.trace.session": session_id,
                }
            )

        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id

        if session_id:
            mlflow.update_current_trace(
                metadata={
                    "mlflow.trace.session": session_id,
                }
            )

        messages = to_chat_completions_input([i.model_dump() for i in request.input])
        if SYSTEM_PROMPT:
            messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        yield from self.call_and_run_tools(messages=messages)


# Log the model using MLflow
mlflow.openai.autolog()
AGENT = ToolCallingAgent(llm_endpoint=LLM_ENDPOINT_NAME, tools=TOOL_INFOS)
mlflow.models.set_model(AGENT)

Overwriting agent.py


#### Test the agent

Interact with the agent to sense-check its output. Since we manually traced methods within `ResponsesAgent`, it is possible to view the trace for each step the agent takes, with any LLM calls made via the OpenAI SDK automatically traced by autologging.

In [0]:
dbutils.library.restartPython()

In [0]:
from agent import AGENT

AGENT.predict(
    {"input": [{"role": "user", "content": "Which income category has the highest average monthly spend?"}], "custom_inputs": {"session_id": "test-session-123"}},
)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-23b729c0-3db9-4763-82eb-510268689ea1/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


ResponsesAgentResponse(tool_choice=None, truncation=None, id=None, created_at=None, error=None, incomplete_details=None, instructions=None, metadata=None, model=None, object='response', output=[OutputItem(type='function_call', id='chatcmpl_72a0ac4c-4d55-44d4-b25e-b5398fb29869_e53af079-8de6-49c7-8be9-ae2ec5485575', call_id='call_a3f21979-bb5f-4873-9964-f98af690a68a', name='analytics__marketing__demographic_profile', arguments='{ "demo_field": "Income_category" }'), OutputItem(type='function_call_output', call_id='call_a3f21979-bb5f-4873-9964-f98af690a68a', output='group_value,avg_spend_per_month,avg_purchases_per_month,avg_recency_days\nHigh,114.15,1.77,48.74\nLow,4.52,0.47,48.38\nMedium,32.57,1.02,49.96\n'), OutputItem(type='message', id='chatcmpl_ef53f95c-1fd8-4ba1-bc78-4f33388ce722_cb937e77-5331-4634-a08d-10b7ef6b82a8', content=[{'text': 'The income category with the highest average monthly spend is High, with an average monthly spend of 114.15, according to the demographic_profile t

Trace(trace_id='tr-fcbd7be1f4899a374b49f2c107718b94')

In [0]:
for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "Which income category has the highest average monthly spend?"}], "custom_inputs": {"session_id": "test-session-123"}}
):
    print(chunk.model_dump(exclude_none=True))

{'type': 'response.output_item.done', 'item': {'type': 'function_call', 'id': 'chatcmpl_b25acecd-2ec5-4393-ba14-8f6464395993_375667a4-8e36-4480-a3e4-481707dda0b3', 'call_id': 'call_04a77869-9c42-41cc-9115-2352a2d16e24', 'name': 'analytics__marketing__demographic_profile', 'arguments': '{ "demo_field": "Income_category" }'}}
{'type': 'response.output_item.done', 'item': {'type': 'function_call_output', 'call_id': 'call_04a77869-9c42-41cc-9115-2352a2d16e24', 'output': 'group_value,avg_spend_per_month,avg_purchases_per_month,avg_recency_days\nHigh,114.15,1.77,48.74\nLow,4.52,0.47,48.38\nMedium,32.57,1.02,49.96\n'}}
{'type': 'response.output_text.delta', 'item_id': 'chatcmpl_43a47748-9523-4fc8-a4c2-64e19cee636a_58fe45cd-8ca9-4816-b1e5-de61664eaa26', 'delta': ''}
{'type': 'response.output_text.delta', 'item_id': 'chatcmpl_43a47748-9523-4fc8-a4c2-64e19cee636a_58fe45cd-8ca9-4816-b1e5-de61664eaa26', 'delta': 'The '}
{'type': 'response.output_text.delta', 'item_id': 'chatcmpl_43a47748-9523-4fc8

Trace(trace_id='tr-6f9d40c574d9855a509b99e6bd903705')

#### Log the `agent` as an MLflow model
Determine Databricks resources to specify for automatic auth passthrough at deployment time. 

Log the agent as code from the `agent.py` file.

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from agent import LLM_ENDPOINT_NAME, uc_toolkit
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from pkg_resources import get_distribution

resources = [DatabricksServingEndpoint(endpoint_name="databricks-meta-llama-3-3-70b-instruct")]
for tool in uc_toolkit.tools:
    udf_name = tool.get("function", {}).get("name", "").replace("__", ".")
    resources.append(DatabricksFunction(function_name=udf_name))

input_example = {
    "input": [
        {
            "role": "user",
            "content": "Which income category has the highest average monthly spend?"
        }
    ],
    "custom_inputs": {
        "session_id": "test-session"
    }
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        input_example=input_example,
        pip_requirements=[
            "databricks-openai",
            "backoff",
            f"databricks-connect=={get_distribution('databricks-connect').version}",
        ],
        resources=resources,
    )

#### Evaluate the agent

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, Safety, Correctness, RetrievalRelevance, RetrievalGroundedness

eval_dataset = [
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Which income category has the highest average monthly spend?"
                }
            ]
        },
               "expectations": {
            "expected_facts": [
                "The income category with the highest average monthly spend is High",
                "The average monthly spend for the High income category is 114.15",
                "The demographic_profile tool was used to determine this"
            ]
        }
    },
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Which income category has the highest average monthly spend?"
                },
                {
                    "role": "assistant",
                    "content": "To determine which income category has the highest average monthly spend, we first need to understand the demographic distribution of our customers. By analyzing the demographic profile, specifically by income category, we can identify which group tends to spend the most on average per month.\n\n<uc_function_call>{\"id\":\"xuqpxgNbZZ9fUxFqLyc5\",\"name\":\"analytics.marketing.demographic_profile\",\"arguments\":\"{\\\"demo_field\\\":\\\"Income_category\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"xuqpxgNbZZ9fUxFqLyc5\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"group_value\\\",\\\"avg_spend_per_month\\\",\\\"avg_purchases_per_month\\\",\\\"avg_recency_days\\\"],\\\"rows\\\":[[\\\"High\\\",114.15,1.77,48.74],[\\\"Low\\\",4.52,0.47,48.38],[\\\"Medium\\\",32.57,1.02,49.96]]}\"}</uc_function_result>\n\nThe income category with the highest average monthly spend is High, with an average monthly spend of 114.15, according to the demographic_profile tool for the Income_category group."
                },
                {
                    "role": "user",
                    "content": "Do graduates use deals more than non-graduates?"
                }
            ]
        },
               "expectations": {
            "expected_facts": [
                "Non-graduates (Basic education) use deals more than graduates",
                "The average deal ratio for Basic education is approximately 0.3505",
                "The average deal ratio for Graduation is approximately 0.2373",
                "The deal_affinity_by_demographic tool was used to determine this"
            ]
        }
    },
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Which income category has the highest average monthly spend?"
                },
                {
                    "role": "assistant",
                    "content": "To determine which income category has the highest average monthly spend, we first need to understand the demographic distribution of our customers. By analyzing the demographic profile, specifically by income category, we can identify which group tends to spend the most on average per month.\n\n<uc_function_call>{\"id\":\"xuqpxgNbZZ9fUxFqLyc5\",\"name\":\"analytics.marketing.demographic_profile\",\"arguments\":\"{\\\"demo_field\\\":\\\"Income_category\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"xuqpxgNbZZ9fUxFqLyc5\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"group_value\\\",\\\"avg_spend_per_month\\\",\\\"avg_purchases_per_month\\\",\\\"avg_recency_days\\\"],\\\"rows\\\":[[\\\"High\\\",114.15,1.77,48.74],[\\\"Low\\\",4.52,0.47,48.38],[\\\"Medium\\\",32.57,1.02,49.96]]}\"}</uc_function_result>\n\nThe income category with the highest average monthly spend is High, with an average monthly spend of 114.15, according to the demographic_profile tool for the Income_category group."
                },
                {
                    "role": "user",
                    "content": "Do graduates use deals more than non-graduates?"
                },
                {
                    "role": "assistant",
                    "content": "To compare the deal affinity of graduates and non-graduates, we need to analyze the deal affinity by education level.\n\n<uc_function_call>{\"id\":\"dLGi6Qm5C2EhENt9iOgg\",\"name\":\"analytics.marketing.deal_affinity_by_demographic\",\"arguments\":\"{\\\"demo_field\\\":\\\"Education\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"dLGi6Qm5C2EhENt9iOgg\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"group_value\\\",\\\"avg_deal_ratio\\\",\\\"avg_deal_count\\\"],\\\"rows\\\":[[\\\"2n Cycle\\\",0.2682,2.2951],[\\\"Basic\\\",0.3505,1.8571],[\\\"Graduation\\\",0.2373,2.2883],[\\\"Master\\\",0.2384,2.3855],[\\\"PhD\\\",0.2277,2.3619]]}\"}</uc_function_result>\n\nAccording to the deal_affinity_by_demographic tool for the Education group, graduates (Graduation category) have an average deal ratio of 0.2373. For comparison, the average deal ratio for non-graduates (Basic category) is 0.3505, which is higher. Therefore, non-graduates use deals more than graduates."
                },
                {
                    "role": "user",
                    "content": "Which at-risk segments have the highest monetary value, and what channel should we use to win them back?"
                }
            ]
        },
                "expectations": {
            "expected_facts": [
                "The at-risk segments with the highest monetary value are High value at risk and Critical retention",
                "High value at risk has an average monetary value of approximately 1177.8",
                "Critical retention has an average monetary value of approximately 1176.17",
                "The store channel has the highest ratio for both segments",
                "Store ratio is approximately 0.4375 for High value at risk and 0.4351 for Critical retention",
                "High value at risk is prioritised over Critical retention based on recency",
                "High value at risk customers are cooling (R=2) while Critical retention customers have lapsed (R=1)"
            ]
        }
    },
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Which income category has the highest average monthly spend?"
                },
                {
                    "role": "assistant",
                    "content": "To determine which income category has the highest average monthly spend, we first need to understand the demographic distribution of our customers. By analyzing the demographic profile, specifically by income category, we can identify which group tends to spend the most on average per month.\n\n<uc_function_call>{\"id\":\"xuqpxgNbZZ9fUxFqLyc5\",\"name\":\"analytics.marketing.demographic_profile\",\"arguments\":\"{\\\"demo_field\\\":\\\"Income_category\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"xuqpxgNbZZ9fUxFqLyc5\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"group_value\\\",\\\"avg_spend_per_month\\\",\\\"avg_purchases_per_month\\\",\\\"avg_recency_days\\\"],\\\"rows\\\":[[\\\"High\\\",114.15,1.77,48.74],[\\\"Low\\\",4.52,0.47,48.38],[\\\"Medium\\\",32.57,1.02,49.96]]}\"}</uc_function_result>\n\nThe income category with the highest average monthly spend is High, with an average monthly spend of 114.15, according to the demographic_profile tool for the Income_category group."
                },
                {
                    "role": "user",
                    "content": "Do graduates use deals more than non-graduates?"
                },
                {
                    "role": "assistant",
                    "content": "To compare the deal affinity of graduates and non-graduates, we need to analyze the deal affinity by education level.\n\n<uc_function_call>{\"id\":\"dLGi6Qm5C2EhENt9iOgg\",\"name\":\"analytics.marketing.deal_affinity_by_demographic\",\"arguments\":\"{\\\"demo_field\\\":\\\"Education\\\"}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"dLGi6Qm5C2EhENt9iOgg\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"group_value\\\",\\\"avg_deal_ratio\\\",\\\"avg_deal_count\\\"],\\\"rows\\\":[[\\\"2n Cycle\\\",0.2682,2.2951],[\\\"Basic\\\",0.3505,1.8571],[\\\"Graduation\\\",0.2373,2.2883],[\\\"Master\\\",0.2384,2.3855],[\\\"PhD\\\",0.2277,2.3619]]}\"}</uc_function_result>\n\nAccording to the deal_affinity_by_demographic tool for the Education group, graduates (Graduation category) have an average deal ratio of 0.2373. For comparison, the average deal ratio for non-graduates (Basic category) is 0.3505, which is higher. Therefore, non-graduates use deals more than graduates."
                },
                {
                    "role": "user",
                    "content": "Which at-risk segments have the highest monetary value, and what channel should we use to win them back?"
                },
                {
                    "role": "assistant",
                    "content": "To identify the at-risk segments with the highest monetary value and determine the best channel for winning them back.\n\n<uc_function_call>{\"id\":\"xe96KEtlSiBYirTQ4pzl\",\"name\":\"analytics.marketing.rfm_segment_summary\",\"arguments\":\"{}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"xe96KEtlSiBYirTQ4pzl\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"Segment\\\",\\\"customer_count\\\",\\\"avg_recency_days\\\",\\\"avg_frequency\\\",\\\"avg_monetary\\\",\\\"avg_r_score\\\",\\\"avg_f_score\\\",\\\"avg_m_score\\\"],\\\"rows\\\":[[\\\"High value at risk\\\",263,50.54,19.94,1177.8,2.0,4.3,4.51],[\\\"Standard base\\\",402,49.2,7.8,155.27,2.0,2.0,1.99],[\\\"Champions\\\",224,16.33,20.81,1167.47,3.0,4.47,4.52],[\\\"Recents\\\",274,15.99,5.2,59.11,3.0,1.42,1.51],[\\\"Low-priority churn\\\",390,82.74,7.67,148.33,1.0,1.97,2.0],[\\\"Critical retention\\\",274,82.86,19.41,1176.17,1.0,4.23,4.48],[\\\"Loyal\\\",106,16.35,12.33,320.14,3.0,3.0,3.0],[\\\"High-potential growth\\\",75,14.65,15.28,914.4,3.0,3.48,3.83]]}\"}</uc_function_result>\n\nTo find the most effective marketing channel to win back High value at risk customers\n\n<uc_function_call>{\"id\":\"LWPYe5dJve9QoOuDZy50\",\"name\":\"analytics.marketing.channel_mix_by_rfm_segment\",\"arguments\":\"{}\"}</uc_function_call>\n\n<uc_function_result>{\"id\":\"LWPYe5dJve9QoOuDZy50\",\"content\":\"{\\\"is_truncated\\\":false,\\\"columns\\\":[\\\"Segment\\\",\\\"avg_web_ratio\\\",\\\"avg_store_ratio\\\",\\\"avg_catalogue_ratio\\\"],\\\"rows\\\":[[\\\"High value at risk\\\",0.2948,0.4375,0.2678],[\\\"Standard base\\\",0.3526,0.5568,0.0905],[\\\"Champions\\\",0.2978,0.4357,0.2666],[\\\"Recents\\\",0.3303,0.6011,0.0686],[\\\"Low-priority churn\\\",0.3541,0.5477,0.0982],[\\\"Critical retention\\\",0.2926,0.4351,0.2723],[\\\"Loyal\\\",0.401,0.4606,0.1384],[\\\"High-potential growth\\\",0.3415,0.4325,0.226]]}\"}</uc_function_result>\n\nThe at-risk segments are High value at risk and Critical retention. They have the highest monetary value, with average monetary values of 1177.8 and 1176.17 respectively, according to the rfm_segment_summary tool. \n\nTo win them back, we should use the store channel, as it has the highest ratio for both segments, with 0.4375 for High value at risk and 0.4351 for Critical retention, according to the channel_mix_by_rfm_segment tool. Prioritizing by recency, we should focus on High value at risk first, as they are still cooling (R=2) and easier to retain than Critical retention, who have already lapsed (R=1)."
                },
                {
                    "role": "user",
                    "content": "What are the characteristics of the Platinum segment?"
                }
            ]
        },
        "expectations": {
            "expected_facts": [
                "No information exists about a 'Platinum' segment",
                "Available segments are Champions, High-potential growth, Loyal, Recents, Critical retention, Low-priority churn, High value at risk, Standard base"
            ]
        }
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input, "custom_inputs": {"session_id": "evaluation-session"}}),
    scorers=[RelevanceToQuery(), Safety(), Correctness()],
)

#### Pre-deployment validation of the agent
Before registering and deploying the agent, we perform pre-deployment checks via the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. 

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Hello!"}], "custom_inputs": {"session_id": "validation-session"}},
    env_manager="uv",
)

#### Register the model to Unity Catalog

In [0]:
mlflow.set_registry_uri("databricks-uc")

# Define the catalog, schema, and model name
catalog = "analytics"
schema = "marketing"
model_name = "marketing-analytics-agent-v2"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

#### Deploy the agent
This agent is within a Databricks Free edition account, so need to use `scale_to_zero=True`.

In [0]:
from databricks import agents
agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, tags = {"endpointSource": "playground"}, scale_to_zero=True)